# Advanced Python: End-to-End Data Acquisition & Cleaning Pipeline

## Exercise 1: Extract SWAPI planet information (single API call)
Write a python script to retrieve information from the swapi API (https://swapi.dev/api/) analogous to the
examples seen.

In [274]:
import requests
import json

BASE_URL = "https://swapi.dev/api/planets/"

def fetch_planet(planet_id: int) -> dict:
    """Fetch one planet from SWAPI and return it as a Python dict."""
    url = f"{BASE_URL}{planet_id}/"
    r = requests.get(url, timeout=20)
    r.raise_for_status()  # raises an error if request failed (404, 500, etc.)
    return r.json()       # JSON -> Python dict

In [275]:
#Collect information about three different planets.

In [276]:
planet_ids = [1, 2, 3]  # Tatooine, Alderaan, Yavin IV (usually)
planets = [fetch_planet(pid) for pid in planet_ids]

# Inspect the received responses in detail:
• What is the format of the data?

• What Python format is suitable to store the information, how are you transforming it?

• What kind of information is provided, how many entries (keys) are in one response?

In [277]:
print("DATA FORMAT CHECK")
print("Type of full response container:", type(planets))        # list
print("Type of one planet entry:", type(planets[0]))            # dict

print("\nKEYS IN ONE RESPONSE")
keys = list(planets[0].keys())
print("Keys:", keys)
print("Number of keys in ONE planet response:", len(keys))

print("\nPRETTY PRINT FIRST PLANET (truncated view)")
print(json.dumps(planets[0], indent=2)[:1000])  # show first 1000 chars

# Optional: transform into a simpler structure (only a few fields)
selected_fields = ["name", "climate", "terrain", "population", "diameter", "gravity"]
planets_clean = [{k: p.get(k) for k in selected_fields} for p in planets]

print("\nTRANSFORMED (selected fields only):")
for p in planets_clean:
    print(p)

DATA FORMAT CHECK
Type of full response container: <class 'list'>
Type of one planet entry: <class 'dict'>

KEYS IN ONE RESPONSE
Keys: ['name', 'rotation_period', 'orbital_period', 'diameter', 'climate', 'gravity', 'terrain', 'surface_water', 'population', 'residents', 'films', 'created', 'edited', 'url']
Number of keys in ONE planet response: 14

PRETTY PRINT FIRST PLANET (truncated view)
{
  "name": "Tatooine",
  "rotation_period": "23",
  "orbital_period": "304",
  "diameter": "10465",
  "climate": "arid",
  "gravity": "1 standard",
  "terrain": "desert",
  "surface_water": "1",
  "population": "200000",
  "residents": [
    "https://swapi.dev/api/people/1/",
    "https://swapi.dev/api/people/2/",
    "https://swapi.dev/api/people/4/",
    "https://swapi.dev/api/people/6/",
    "https://swapi.dev/api/people/7/",
    "https://swapi.dev/api/people/8/",
    "https://swapi.dev/api/people/9/",
    "https://swapi.dev/api/people/11/",
    "https://swapi.dev/api/people/43/",
    "https://sw

## Answer 

### What is the format of the data?

- The API returns JSON, which becomes a Python dictionary. More specifically a list of dictionaries for multiple planets. 

### What Python format is suitable to store the information, how are you transforming it?

- dict, list, optional transformation (e.g., selecting specific keys) are suitable for transforming the data.

### What kind of information is provided, how many entries (keys) are in one response?

- you receive 3 API responses (for 3 planets), each response contains ~14–16 keys (depending on SWAPI version).

### Observation 

All numeric values like:

"rotation_period": "23"

"diameter": "10465"

"population": "200000"

are actually stored as strings, not integers! 

## Exercise 2: Store SWAPI Information in Pandas Data Frame (pagination)

Combine information from all planets in one single DataFrame.

Get information about how many planets there are:
data = request.get("https://swapi.dev/api/planets")

# How to Pagination  

https://swapi.dev/api/planets/ is paginated.

That means:

- It returns only 10 planets per page

- There is a "next" field in the JSON

- You must loop until "next" is None

If you don’t handle pagination → you only get 10 planets

In [278]:
import requests
import pandas as pd

BASE_URL = "https://swapi.dev/api/planets/"

In [279]:
# Get total count 

response = requests.get(BASE_URL)
data = response.json()

print("Total number of planets:", data["count"])

Total number of planets: 60


In [280]:
# Create empty DataFrame

df = pd.DataFrame(columns=["name", "diameter", "population"])

In [281]:
# Create an empty DataFrame with the keys you are interested to extract as columns.
# e.g. df = pd.DataFrame(columns = ['name', 'diameter', 'population'])

url = BASE_URL

while url:
    response = requests.get(url)
    data = response.json()
    
    for planet in data["results"]:
        df.loc[len(df)] = [
            planet["name"],
            planet["diameter"],
            planet["population"]
        ]
    
    url = data["next"]  # move to next page

In [282]:
# Convert numeric columns (optional but recommended)

df["diameter"] = pd.to_numeric(df["diameter"], errors="coerce")
df["population"] = pd.to_numeric(df["population"], errors="coerce")

In [283]:
# Loop over all planets and append the retrieved information to your data frame. 
# Use df.info() and df.describe() to review your result

print("\nDataFrame Info:")
df.info()

print("\nDescriptive Statistics:")
print(df.describe())

df.head()


DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
Index: 60 entries, 0 to 59
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   name        60 non-null     object 
 1   diameter    44 non-null     float64
 2   population  43 non-null     float64
dtypes: float64(2), object(1)
memory usage: 1.9+ KB

Descriptive Statistics:
            diameter    population
count      44.000000  4.300000e+01
mean    12388.340909  3.980003e+10
std     17045.948720  1.685799e+11
min         0.000000  0.000000e+00
25%      7812.250000  1.050000e+07
50%     11015.000000  5.000000e+08
75%     13422.500000  3.750000e+09
max    118000.000000  1.000000e+12


,name,diameter,population
0,Tatooine,10465.0,2.000000e+05
1,Alderaan,12500.0,2.000000e+09
2,Yavin IV,10200.0,1.000000e+03
3,Hoth,7200.0,NaN
4,Dagobah,8900.0,NaN


# Interpretation of Result
The resulting DataFrame contains 60 planets with three selected variables: name, diameter, and population. After converting numeric fields from strings to float values, diameter and population are stored as float64, while name remains an object (string). Missing values occur due to “unknown” entries in the original API data, which were converted to NaN. Descriptive statistics indicate high variability in both diameter and population. In particular, population shows strong right-skewness, as the mean significantly exceeds the median, suggesting the presence of extreme outliers.

In [284]:
# Cleaner Version 

all_planets = []

url = BASE_URL
while url:
    response = requests.get(url)
    data = response.json()
    all_planets.extend(data["results"])
    url = data["next"]

df = pd.DataFrame(all_planets)[["name", "diameter", "population"]]
df["diameter"] = pd.to_numeric(df["diameter"], errors="coerce")
df["population"] = pd.to_numeric(df["population"], errors="coerce")

# Exercise 3: Data manipulation and cleaning exercise

1. Load the provided data set from:
https://raw.githubusercontent.com/DJCordhose/ml-examples/master/datasets/Iris/iris_dirty.csv
Hint: pandas also provides a method to download data set directly from the web.

2. Inspect the data set with appropriate functions from pandas and numpy and reveal irregularities.

   
3. Clean impurities with suitable methods by applying discussed “rules” for data cleaning.

In [287]:
import pandas as pd
import numpy as np

# ----------------------------
# 1) LOAD (robustly)
# ----------------------------
url = "https://raw.githubusercontent.com/DJCordhose/ml-examples/master/datasets/Iris/iris_dirty.csv"

# This dataset can be messy: load with header=None so the first row is not treated as column names
df = pd.read_csv(url, header=None)

# Assign correct column names manually
df.columns = ["sepal_length", "sepal_width", "petal_length", "petal_width", "species"]

# ----------------------------
# 2) CLEAN CATEGORICAL COLUMN (species)
# ----------------------------
# Standardize formatting (strip spaces, lowercase, unify separators)
df["species"] = (
    df["species"]
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(" ", "-", regex=False)
)

# Fix known typo(s)
df["species"] = df["species"].replace({
    "iris-setsoa": "iris-setosa"
})

# ----------------------------
# 3) CLEAN NUMERIC COLUMNS
# ----------------------------
num_cols = ["sepal_length", "sepal_width", "petal_length", "petal_width"]

# Remove units like " mm" if they appear, then convert to numeric
for c in num_cols:
    df[c] = (
        df[c]
        .astype(str)
        .str.replace(" mm", "", regex=False)
        .str.strip()
    )
    df[c] = pd.to_numeric(df[c], errors="coerce")  # invalid -> NaN

# ----------------------------
# 4) HANDLE MISSING VALUES
# ----------------------------
# Option chosen: drop rows with missing values (simple + acceptable here)
df = df.dropna()

# ----------------------------
# 5) REMOVE IMPOSSIBLE VALUES
# ----------------------------
# Iris measurements should be > 0
df = df[(df[num_cols] > 0).all(axis=1)]

# ----------------------------
# 6) REMOVE DUPLICATES
# ----------------------------
df = df.drop_duplicates()

# ----------------------------
# 7) FINAL VALIDATION
# ----------------------------
print("Missing values per column:")
print(df.isna().sum(), "\n")

print("Dtypes:")
print(df.dtypes, "\n")

print("Species distribution:")
print(df["species"].value_counts(), "\n")

print("Summary statistics:")
print(df.describe(), "\n")

df.head()

Missing values per column:
sepal_length    0
sepal_width     0
petal_length    0
petal_width     0
species         0
dtype: int64 

Dtypes:
sepal_length    float64
sepal_width     float64
petal_length    float64
petal_width       int64
species          object
dtype: object 

Species distribution:
species
iris-virginica     50
iris-versicolor    49
iris-setosa        48
Name: count, dtype: int64 

Summary statistics:
       sepal_length  sepal_width  petal_length  petal_width
count    147.000000   147.000000    147.000000   147.000000
mean       6.211565     3.055782      3.788435    12.136054
std        4.379881     0.437009      1.762451     7.600144
min        4.300000     2.000000      1.000000     1.000000
25%        5.100000     2.800000      1.600000     3.000000
50%        5.800000     3.000000      4.400000    13.000000
75%        6.400000     3.300000      5.100000    18.000000
max       58.000000     4.400000      6.900000    25.000000 



,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,2,iris-setosa
1,4.9,3.0,1.4,2,iris-setosa
2,4.7,3.2,1.3,2,iris-setosa
3,4.6,3.1,1.5,2,iris-setosa
4,5.0,3.6,1.4,2,iris-setosa


## Exercise 4: Traffic_Manipulated: Data manipulation and cleaning exercise (script / jupyter / marimo)

1. Load the provided traffic_manipulated.csv data set from ILIAS and print the first 2 rows with together with the
column names. (-> df = pd.read_csv("./traffic_manipulated.csv", index_col='index'))

In [288]:
# load dataset 

import pandas as pd
import numpy as np

df = pd.read_csv("./data/traffic_manipulated.csv", index_col="index")

df.head(2)

FileNotFoundError: [Errno 2] No such file or directory: './data/traffic_manipulated.csv'

In [ ]:
# How Many NaNs in Entire DataFrame?

df.isna().sum().sum()

# quick tip: 

if output does not immediately run in jupyter press **Shift + Enter**.

In [ ]:
#Mean and Max of Numeric Columns (dtypes, then compute)

df.dtypes
df.select_dtypes(include=np.number).agg(["mean", "max"])

### Check If All Numeric Columns Are Really Numeric

Based on column names + first rows:

- some numeric-looking columns stored as object

- maybe strings like "2400" or "124 clicks"

In [ ]:
df.head(2)
df.dtypes

In [ ]:
#Fix Problematic Columns
